# 🚨 Emergency Vehicle Detection — Colab Pro (A100)

ทำตามขั้นตอนตามลำดับ **Run All** หรือรันทีละ Cell
GPU: **A100** · Framework: **FastAPI + YOLOv8 (Ultralytics)**


## 1️⃣  ตรวจสอบ GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


## 2️⃣  ติดตั้ง dependencies

In [ ]:
!pip install -q fastapi uvicorn[standard] ultralytics opencv-python-headless pyngrok python-multipart jinja2 aiofiles
print("✅ ติดตั้งเสร็จ")


## 3️⃣  สร้าง project structure

In [ ]:
import os
BASE = '/content/evd'
for d in ['models', 'videos', 'templates', 'static']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
print("✅ Directories ready:", BASE)


## 4️⃣  เขียนไฟล์ Python

In [ ]:
from pathlib import Path
Path('/content/evd/config.py').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/config.py').write_text('''"""config.py — ตั้งค่าทั้งหมดของระบบ (Colab A100 version)"""
from pathlib import Path

BASE_DIR = Path(__file__).parent

# ===== โมเดล =====
MODEL_PATH_X   = BASE_DIR / "models" / "best_x.pt"
MODEL_PATH_M   = BASE_DIR / "models" / "best_m.pt"
FALLBACK_X     = "yolov8x.pt"
FALLBACK_M     = "yolov8m.pt"
MODEL_PATH     = MODEL_PATH_X
FALLBACK_MODEL = FALLBACK_X

# ===== กล้อง 4 มุม =====
CAMERA_VIDEOS = {
    "cam1": BASE_DIR / "videos" / "north.mp4",
    "cam2": BASE_DIR / "videos" / "east.mp4",
    "cam3": BASE_DIR / "videos" / "south.mp4",
    "cam4": BASE_DIR / "videos" / "west.mp4",
}
CAMERA_LABELS = {
    "cam1": "CAM-1 · NORTH",
    "cam2": "CAM-2 · EAST",
    "cam3": "CAM-3 · SOUTH",
    "cam4": "CAM-4 · WEST",
}

# ===== พารามิเตอร์การตรวจจับ — ปรับสำหรับ A100 =====
CONF_THRESHOLD = 0.40
IMG_SIZE       = 640    # A100 รองรับ full resolution
DEVICE         = "cuda" # ใช้ GPU

TARGET_FPS        = 20
EMERGENCY_CLASSES = {"ambulance", "firetruck", "police"}
''', encoding='utf-8')
print('✅ /content/evd/config.py')

In [ ]:
from pathlib import Path
Path('/content/evd/detector.py').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/detector.py').write_text('''"""
detector.py — โหลดโมเดล YOLOv8 และตรวจจับยานพาหนะฉุกเฉิน
รองรับ 2 โมเดล: yolov8x (แม่นกว่า) และ yolov8m (เร็วกว่า)
โหลดครั้งเดียวตอน startup ไม่โหลดซ้ำทุก request
"""
import time
import threading
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

import config

CLASS_COLORS = {
    "ambulance": (53, 107, 255),
    "firetruck": (53, 53, 224),
    "police":    (255, 158, 75),
}
DEFAULT_COLOR = (122, 196, 0)

MODEL_INFO = {
    "x": {"name": "YOLOv8x", "params": "68.2M parameters"},
    "m": {"name": "YOLOv8m", "params": "25.9M parameters"},
}


class Detector:
    def __init__(self):
        self._models: dict = {}
        self._lock = threading.Lock()
        self.last_infer_ms = 0.0
        self.using_custom  = False

        if Path(config.MODEL_PATH_X).exists():
            self._models["x"] = YOLO(str(config.MODEL_PATH_X))
            self.using_custom  = True
            print(f"[Detector] โหลด YOLOv8x custom: {config.MODEL_PATH_X}")
        else:
            self._models["x"] = YOLO(config.FALLBACK_X)
            print(f"[Detector] ไม่พบ best_x.pt → ใช้ pretrained {config.FALLBACK_X}")

        if Path(config.MODEL_PATH_M).exists():
            self._models["m"] = YOLO(str(config.MODEL_PATH_M))
            self.using_custom  = True
            print(f"[Detector] โหลด YOLOv8m custom: {config.MODEL_PATH_M}")
        else:
            self._models["m"] = YOLO(config.FALLBACK_M)
            print(f"[Detector] ไม่พบ best_m.pt → ใช้ pretrained {config.FALLBACK_M}")

        self.model = self._models["x"]
        self.names = self.model.names

    def get_model(self, key: str):
        return self._models.get(key, self._models["x"])

    def get_model_info(self, key: str) -> dict:
        return MODEL_INFO.get(key, MODEL_INFO["x"])

    def color_for(self, name: str):
        return CLASS_COLORS.get(name.lower(), DEFAULT_COLOR)

    def infer(self, frame, model_key: str = "x", conf=None):
        if conf is None:
            conf = config.CONF_THRESHOLD
        model = self.get_model(model_key)
        t0 = time.time()
        with self._lock:
            results = model.predict(
                frame,
                imgsz=config.IMG_SIZE,
                conf=conf,
                device=config.DEVICE,
                verbose=False,
            )
        self.last_infer_ms = (time.time() - t0) * 1000
        dets  = []
        names = model.names
        r     = results[0]
        if r.boxes is not None:
            for box in r.boxes:
                cls_id   = int(box.cls[0])
                conf_val = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                dets.append({
                    "name": names.get(cls_id, str(cls_id)),
                    "conf": conf_val,
                    "box":  (x1, y1, x2, y2),
                })
        return dets

    def draw(self, frame, dets):
        out = frame.copy()
        for d in dets:
            x1, y1, x2, y2 = d["box"]
            color = self.color_for(d["name"])
            label = f'{d["name"].upper()} {int(d["conf"] * 100)}%'
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(out, (x1, y1 - th - 8), (x1 + tw + 8, y1), color, -1)
            cv2.putText(out, label, (x1 + 4, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        return out


def placeholder_frame(text="NO SIGNAL", w=640, h=360):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:] = (20, 16, 10)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
    cv2.putText(img, text, ((w - tw) // 2, (h + th) // 2),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (80, 100, 140), 2, cv2.LINE_AA)
    return img
''', encoding='utf-8')
print('✅ /content/evd/detector.py')

In [ ]:
from pathlib import Path
Path('/content/evd/state.py').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/state.py').write_text('''"""state.py — เก็บสถานะรวมของระบบ (thread-safe)"""
import time
import threading
from collections import deque

import config


class SystemState:
    def __init__(self):
        self._lock        = threading.Lock()
        self.signals      = {cam: "STOP" for cam in config.CAMERA_VIDEOS}
        self.counts       = {}
        self.log          = deque(maxlen=15)
        self._active      = {cam: set() for cam in config.CAMERA_VIDEOS}
        self.infer_ms     = 0.0
        self.current_model = "x"
        self.current_conf  = config.CONF_THRESHOLD

    def set_model(self, key: str):
        with self._lock:
            self.current_model = key if key in ("x", "m") else "x"

    def set_conf(self, conf: float):
        with self._lock:
            self.current_conf = max(0.0, min(1.0, conf))

    def update(self, cam_id, dets):
        emerg_now = {
            d["name"].lower() for d in dets
            if d["name"].lower() in config.EMERGENCY_CLASSES
        }
        with self._lock:
            prev         = self._active.get(cam_id, set())
            new_arrivals = emerg_now - prev
            for name in new_arrivals:
                self.counts[name] = self.counts.get(name, 0) + 1
                conf = max(
                    (d["conf"] for d in dets if d["name"].lower() == name),
                    default=0,
                )
                self.log.appendleft({
                    "name": name,
                    "cam":  config.CAMERA_LABELS.get(cam_id, cam_id),
                    "conf": int(conf * 100),
                    "t":    time.strftime("%H:%M:%S"),
                })
            self._active[cam_id] = emerg_now
            self.signals[cam_id] = "CLEAR" if emerg_now else "STOP"

    def set_infer_ms(self, ms):
        with self._lock:
            self.infer_ms = ms

    def snapshot(self):
        with self._lock:
            return {
                "signals":       dict(self.signals),
                "counts":        dict(self.counts),
                "log":           list(self.log),
                "infer_ms":      round(self.infer_ms, 1),
                "current_model": self.current_model,
                "current_conf":  round(self.current_conf, 2),
            }


state = SystemState()
''', encoding='utf-8')
print('✅ /content/evd/state.py')

In [ ]:
from pathlib import Path
Path('/content/evd/camera.py').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/camera.py').write_text('''"""camera.py — ตัวจัดการกล้องแต่ละตัว (2 threads ต่อกล้อง)"""
import time
import threading

import cv2

import config
from detector import placeholder_frame
from state import state


class CameraWorker:
    def __init__(self, cam_id, detector):
        self.cam_id     = cam_id
        self.detector   = detector
        self.video_path = config.CAMERA_VIDEOS.get(cam_id)
        self.boxes       = []
        self.current_raw = None
        self.latest_jpeg = None
        self._lock   = threading.Lock()
        self.running = True

    def start(self):
        if not self.video_path or not self.video_path.exists():
            ph = placeholder_frame(f"NO SIGNAL · {self.cam_id}")
            ok, buf = cv2.imencode(".jpg", ph)
            if ok:
                with self._lock:
                    self.latest_jpeg = buf.tobytes()
            return
        threading.Thread(target=self._playback_loop,  daemon=True).start()
        threading.Thread(target=self._inference_loop, daemon=True).start()

    def _playback_loop(self):
        cap   = cv2.VideoCapture(str(self.video_path))
        delay = 1.0 / max(config.TARGET_FPS, 1)
        while self.running:
            ok, frame = cap.read()
            if not ok:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
            self.current_raw = frame
            annotated = self.detector.draw(frame, self.boxes)
            ok2, buf  = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, 80])
            if ok2:
                with self._lock:
                    self.latest_jpeg = buf.tobytes()
            time.sleep(delay)

    def _inference_loop(self):
        while self.running:
            frame = self.current_raw
            if frame is None:
                time.sleep(0.01)
                continue
            dets = self.detector.infer(
                frame,
                model_key=state.current_model,
                conf=state.current_conf,
            )
            self.boxes = dets
            state.update(self.cam_id, dets)
            state.set_infer_ms(self.detector.last_infer_ms)
            time.sleep(0.005)

    def get_jpeg(self):
        with self._lock:
            return self.latest_jpeg
''', encoding='utf-8')
print('✅ /content/evd/camera.py')

In [ ]:
from pathlib import Path
Path('/content/evd/main.py').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/main.py').write_text('''"""main.py — FastAPI server (Colab version)"""
import base64
import os
import tempfile
import time

import cv2
import numpy as np
from fastapi import FastAPI, Request, UploadFile, File, Form
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates

import config
from detector import Detector
from camera import CameraWorker
from state import state

app = FastAPI(title="Emergency Vehicle Detection")
app.mount("/static",    StaticFiles(directory="static"),    name="static")
templates = Jinja2Templates(directory="templates")

detector = Detector()
workers  = {cam: CameraWorker(cam, detector) for cam in config.CAMERA_VIDEOS}
for w in workers.values():
    w.start()


@app.get("/")
def landing(request: Request):
    return templates.TemplateResponse(request, "landing.html", {})

@app.get("/dashboard")
def dashboard(request: Request):
    return templates.TemplateResponse(
        request, "index.html",
        {"cameras": config.CAMERA_LABELS, "using_custom": detector.using_custom},
    )

@app.get("/detect")
def detect_page(request: Request):
    return templates.TemplateResponse(request, "detect.html", {})

@app.get("/stream/{cam_id}")
def stream(cam_id: str):
    worker = workers.get(cam_id)
    delay  = 1.0 / max(config.TARGET_FPS, 1)
    def gen():
        while True:
            jpg = worker.get_jpeg() if worker else None
            if jpg:
                yield (b"--frame\\r\\nContent-Type: image/jpeg\\r\\n\\r\\n" + jpg + b"\\r\\n")
            time.sleep(delay)
    return StreamingResponse(gen(), media_type="multipart/x-mixed-replace; boundary=frame")

@app.get("/stats")
def stats():
    snap = state.snapshot()
    snap["model_info"] = detector.get_model_info(snap["current_model"])
    return JSONResponse(snap)

@app.get("/api/settings")
def get_settings():
    return JSONResponse({"model": state.current_model, "conf": round(state.current_conf, 2)})

@app.post("/api/settings")
async def post_settings(request: Request):
    body = await request.json()
    if "model" in body: state.set_model(str(body["model"]))
    if "conf"  in body: state.set_conf(float(body["conf"]))
    return JSONResponse({"ok": True})

@app.post("/predict/image")
async def predict_image(
    file: UploadFile = File(...),
    model_name: str  = Form("x"),
    conf: float      = Form(0.25),
):
    data  = await file.read()
    frame = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        return JSONResponse({"error": "ไม่สามารถอ่านไฟล์ภาพได้"}, status_code=400)
    dets      = detector.infer(frame, model_key=model_name, conf=conf)
    annotated = detector.draw(frame, dets)
    _, buf    = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, 85])
    info      = detector.get_model_info(model_name)
    return JSONResponse({
        "image":        base64.b64encode(buf.tobytes()).decode(),
        "detections":   [{"name": d["name"], "conf": round(d["conf"], 3), "box": list(d["box"])} for d in dets],
        "infer_ms":     round(detector.last_infer_ms, 1),
        "model_name":   info["name"],
        "model_params": info["params"],
    })

@app.post("/predict/video")
async def predict_video(
    file: UploadFile = File(...),
    model_name: str  = Form("x"),
    conf: float      = Form(0.25),
):
    data   = await file.read()
    suffix = os.path.splitext(file.filename or ".mp4")[1] or ".mp4"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(data)
        tmp_path = tmp.name
    summary, sample_b64 = {}, None
    try:
        cap, frame_count, t_start = cv2.VideoCapture(tmp_path), 0, time.time()
        while True:
            ok, frame = cap.read()
            if not ok: break
            dets = detector.infer(frame, model_key=model_name, conf=conf)
            frame_count += 1
            for d in dets:
                n = d["name"].lower()
                summary[n] = summary.get(n, 0) + 1
            if dets and sample_b64 is None:
                _, buf    = cv2.imencode(".jpg", detector.draw(frame, dets), [cv2.IMWRITE_JPEG_QUALITY, 80])
                sample_b64 = base64.b64encode(buf.tobytes()).decode()
        cap.release()
        total_ms = (time.time() - t_start) * 1000
        info     = detector.get_model_info(model_name)
        return JSONResponse({
            "frame_count": frame_count, "total_ms": round(total_ms, 1),
            "avg_ms_per_frame": round(total_ms / max(frame_count, 1), 1),
            "detections_summary": summary, "sample_frame": sample_b64,
            "model_name": info["name"], "model_params": info["params"],
        })
    finally:
        os.unlink(tmp_path)

@app.post("/predict/webcam_frame")
async def predict_webcam_frame(request: Request):
    body  = await request.json()
    raw   = base64.b64decode(body.get("image", "").split(",")[-1])
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        return JSONResponse({"error": "ถอดรหัสภาพไม่ได้"}, status_code=400)
    model_name = body.get("model_name", "m")
    conf       = float(body.get("conf", 0.25))
    dets       = detector.infer(frame, model_key=model_name, conf=conf)
    info       = detector.get_model_info(model_name)
    return JSONResponse({
        "detections": [{"name": d["name"], "conf": round(d["conf"], 3), "box": list(d["box"])} for d in dets],
        "infer_ms":   round(detector.last_infer_ms, 1),
        "model_name": info["name"], "model_params": info["params"],
    })
''', encoding='utf-8')
print('✅ /content/evd/main.py')

## 5️⃣  เขียน HTML Templates

In [ ]:
from pathlib import Path
Path('/content/evd/templates/landing.html').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/templates/landing.html').write_text('''<!DOCTYPE html>
<html lang="lo">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Emergency Vehicle Detection System</title>
  <link rel="stylesheet" href="/static/landing.css">
</head>
<body>
  <header class="topbar">
    <div class="topbar-logo">
      <div class="topbar-logo-text">
        <div class="logo-name">ມະຫາວິທະຍາໄລແຫ່ງຊາດ</div>
        <div class="logo-dept">ຄະນະວິທະຍາສາດທຳມະຊາດ · ວິທະຍາສາດຄອມພິວເຕີ</div>
      </div>
    </div>
    <div class="topbar-badge">FYP 2025–26</div>
  </header>
  <main>
    <section class="hero">
      <div class="hero-eyebrow">Real-time · YOLOv8 · Intelligent Signaling</div>
      <h1 class="hero-title">Emergency Vehicle<br><span>Detection System</span></h1>
      <p class="hero-desc">
        ລະບົບກວດຈັບລົດສຸກເສີນແບບ Real-time ດ້ວຍ YOLOv8<br>
        ເພື່ອປັບສັນຍານໄຟຈາລາຈອນໂດຍອັດຕະໂນມັດ
      </p>
      <div style="display:flex;gap:12px;justify-content:center;flex-wrap:wrap">
        <a href="/dashboard" class="cta-btn">Dashboard Monitor &nbsp;→</a>
        <a href="/detect" class="cta-btn" style="background:#16a34a">Detection Tool &nbsp;→</a>
      </div>
    </section>
    <div class="features">
      <div class="feat-card">
        <div class="feat-icon">🚨</div>
        <div class="feat-title">ກວດຈັບ 3 ປະເພດ</div>
        <div class="feat-desc">YOLOv8 ກວດຈັບລົດສຸກເສີນໄດ້ທັນທີ ດ້ວຍຄວາມຖືກຕ້ອງສູງ</div>
        <div class="feat-tags">
          <span class="tag amb">Ambulance</span>
          <span class="tag fire">Firetruck</span>
          <span class="tag police">Police</span>
        </div>
      </div>
      <div class="feat-card">
        <div class="feat-icon">📹</div>
        <div class="feat-title">ມໍນິເຕີ 4 ມຸມກ້ອງ</div>
        <div class="feat-desc">ສະແດງກ້ອງ 4 ມຸມຂອງສີ່ແຍກ ພ້ອມ overlay ຜົນການກວດຈັບແບບ Live</div>
        <div class="feat-tags">
          <span class="tag green">Live Stream</span>
          <span class="tag">2×2 Grid</span>
        </div>
      </div>
      <div class="feat-card">
        <div class="feat-icon">🔬</div>
        <div class="feat-title">Detection Tool</div>
        <div class="feat-desc">ທົດສອບໂມເດລກັບຮູບ/ວີດີໂອ/Webcam ພ້ອມ FPS real-time</div>
        <div class="feat-tags">
          <span class="tag green">YOLOv8x</span>
          <span class="tag">YOLOv8m</span>
        </div>
      </div>
    </div>
  </main>
  <footer>NUOL &nbsp;·&nbsp; Computer Science &nbsp;·&nbsp; FYP 2025–26</footer>
</body>
</html>
''', encoding='utf-8')
print('✅ /content/evd/templates/landing.html')

In [ ]:
from pathlib import Path
Path('/content/evd/templates/index.html').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/templates/index.html').write_text('''<!DOCTYPE html>
<html lang="lo">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>EVD Traffic Control System</title>
  <link rel="stylesheet" href="/static/style.css?v=4">
</head>
<body>
  <div class="dash">
    <header class="topbar">
      <div class="topbar-left">
        <div class="logo-dot"></div>
        <div>
          <div class="logo-text">EVD TRAFFIC CONTROL SYSTEM</div>
          <div class="logo-sub">Emergency Vehicle Detection · <span id="modelLabel">YOLOv8x</span></div>
        </div>
      </div>
      <div class="topbar-right">
        {% if not using_custom %}
        <div class="status-badge warn">PRETRAINED MODEL</div>
        {% endif %}
        <div class="ctrl-inline">
          <label class="ctrl-inline-label">Model</label>
          <select id="modelSelect" class="ctrl-inline-select">
            <option value="x">YOLOv8x · แม่น</option>
            <option value="m">YOLOv8m · เร็ว</option>
          </select>
        </div>
        <div class="ctrl-inline conf-ctrl">
          <label class="ctrl-inline-label">Conf <span id="confDisplay">0.40</span></label>
          <input type="range" id="confSlider" min="0" max="1" step="0.05" value="0.40" class="ctrl-inline-slider">
        </div>
        <div class="status-badge">SYSTEM ACTIVE</div>
        <a href="/detect" class="nav-pill">Detection Tool →</a>
        <div class="clock" id="clk">--:--:--</div>
      </div>
    </header>
    <div class="main">
      <div class="cam-grid">
        {% for cam_id, label in cameras.items() %}
        <div class="cam-cell">
          <img class="cam-feed" src="/stream/{{ cam_id }}" alt="{{ label }}">
          <div class="cam-label">{{ label }}</div>
          <div class="cam-corner"><span class="rec-dot"></span><span class="rec-text">REC</span></div>
        </div>
        {% endfor %}
      </div>
      <aside class="sidebar">
        <section class="sidebar-section">
          <div class="sidebar-title">ສັນຍານໄຟຈາລາຈອນ</div>
          <div class="traffic-grid" id="trafficGrid">
            {% for cam_id, label in cameras.items() %}
            <div class="traffic-light" data-cam="{{ cam_id }}">
              <div class="tl-label">{{ label.split("·")[1] if "·" in label else label }}</div>
              <div class="tl-box">
                <div class="tl-bulb r"></div>
                <div class="tl-bulb y"></div>
                <div class="tl-bulb g"></div>
              </div>
              <div class="tl-status stop">STOP</div>
            </div>
            {% endfor %}
          </div>
        </section>
        <section class="sidebar-section grow">
          <div class="sidebar-title">ບັນທຶກການກວດຈັບ</div>
          <div class="alert-list" id="alertList">
            <div class="empty-hint">ລໍຖ້າການກວດຈັບ...</div>
          </div>
        </section>
        <section class="sidebar-section">
          <div class="sidebar-title">ສະຖິຕິ Session</div>
          <div class="stat-row">
            <div class="stat-card"><div class="stat-num amb" id="cntAmb">0</div><div class="stat-lbl">AMBU</div></div>
            <div class="stat-card"><div class="stat-num fire" id="cntFire">0</div><div class="stat-lbl">FIRE</div></div>
            <div class="stat-card"><div class="stat-num police" id="cntPolice">0</div><div class="stat-lbl">POLICE</div></div>
          </div>
        </section>
      </aside>
    </div>
    <footer class="bottombar">
      <span class="bb-dot"></span>
      <span>Model: <span class="model-tag" id="bottomModel">YOLOv8x</span></span>
      <span class="bb-sep">·</span>
      <span id="bottomParams" class="bb-params">68.2M parameters</span>
      <span class="bb-sep">·</span>
      <span>Inference: <span id="inf">--</span></span>
      <span class="bb-sep">·</span>
      <span>NUOL · Computer Science · FYP 2025–26</span>
    </footer>
  </div>
  <script src="/static/app.js?v=2"></script>
</body>
</html>
''', encoding='utf-8')
print('✅ /content/evd/templates/index.html')

In [ ]:
from pathlib import Path
Path('/content/evd/templates/detect.html').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/templates/detect.html').write_text('''<!DOCTYPE html>
<html lang="th">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>EVD · Detection Tool</title>
  <link rel="stylesheet" href="/static/style.css?v=4">
  <link rel="stylesheet" href="/static/detect.css?v=1">
</head>
<body>
  <div class="dash detect-dash">
    <header class="topbar">
      <div class="topbar-left">
        <div class="logo-dot"></div>
        <div>
          <div class="logo-text">EVD DETECTION TOOL</div>
          <div class="logo-sub">Emergency Vehicle Detection · YOLOv8</div>
        </div>
      </div>
      <div class="topbar-right">
        <a href="/dashboard" class="nav-pill">← Dashboard</a>
        <div class="clock" id="clk">--:--:--</div>
      </div>
    </header>
    <div class="detect-controls">
      <div class="detect-ctrl-group">
        <label class="detect-ctrl-label">โมเดล</label>
        <select id="modelSelect" class="detect-ctrl-select">
          <option value="x">YOLOv8x · แม่นกว่า (68.2M params)</option>
          <option value="m" selected>YOLOv8m · เร็วกว่า (25.9M params)</option>
        </select>
      </div>
      <div class="detect-ctrl-group">
        <label class="detect-ctrl-label">
          Confidence: <span id="confVal" class="conf-badge">0.25</span>
        </label>
        <input type="range" id="confSlider" min="0" max="1" step="0.01" value="0.25" class="detect-slider">
      </div>
    </div>
    <div class="tab-bar">
      <button class="tab-btn active" data-tab="image">🖼️ รูปภาพ</button>
      <button class="tab-btn" data-tab="video">🎬 วิดีโอ</button>
      <button class="tab-btn" data-tab="webcam">📹 Webcam Real-time</button>
    </div>
    <div class="tab-body">
      <div class="tab-pane active" id="tab-image">
        <div class="upload-zone" id="imageDropZone">
          <input type="file" id="imageFile" accept="image/*" hidden>
          <div class="upload-icon">🖼️</div>
          <div class="upload-text">คลิกหรือลากไฟล์ภาพมาวางที่นี่</div>
          <div class="upload-hint">รองรับ JPG · PNG · WEBP</div>
        </div>
        <div class="tab-actions">
          <button class="action-btn" id="imageDetectBtn" disabled>ตรวจจับ</button>
        </div>
        <div id="imageResult" class="result-area hidden">
          <div class="result-meta" id="imageMeta"></div>
          <div class="result-layout">
            <img id="resultImg" class="result-img" alt="ผลตรวจจับ">
            <div class="det-list" id="imageDetList"></div>
          </div>
        </div>
      </div>
      <div class="tab-pane hidden" id="tab-video">
        <div class="upload-zone" id="videoDropZone">
          <input type="file" id="videoFile" accept="video/*" hidden>
          <div class="upload-icon">🎬</div>
          <div class="upload-text">คลิกหรือลากไฟล์วิดีโอมาวางที่นี่</div>
          <div class="upload-hint">รองรับ MP4 · AVI · MOV</div>
        </div>
        <div class="tab-actions">
          <button class="action-btn" id="videoDetectBtn" disabled>ประมวลผลวิดีโอ</button>
        </div>
        <div id="videoProgress" class="progress-row hidden">
          <span class="spinner"></span>
          <span id="progressText">กำลังประมวลผล...</span>
        </div>
        <div id="videoResult" class="result-area hidden">
          <div class="result-meta" id="videoMeta"></div>
          <div class="video-stats" id="videoStats"></div>
          <div id="videoSampleWrap" class="hidden">
            <div class="sample-label">ตัวอย่าง frame ที่พบรถฉุกเฉิน</div>
            <img id="videoSampleImg" class="result-img" alt="ตัวอย่าง">
          </div>
        </div>
      </div>
      <div class="tab-pane hidden" id="tab-webcam">
        <div class="webcam-wrap">
          <div class="webcam-view" id="webcamView">
            <video id="webcamVideo" autoplay muted playsinline></video>
            <canvas id="webcamCanvas"></canvas>
            <div class="wcam-overlay fps-overlay" id="fpsOverlay">-- FPS</div>
            <div class="wcam-overlay model-overlay" id="modelOverlay">--</div>
          </div>
          <div class="webcam-actions">
            <button class="action-btn" id="startWebcam">▶ เริ่มกล้อง</button>
            <button class="action-btn danger" id="stopWebcam" disabled>■ หยุดกล้อง</button>
          </div>
          <div id="webcamError" class="error-box hidden"></div>
          <div id="webcamHint" class="hint-box hidden">
            💡 FPS ต่ำเกินไป? ลองเปลี่ยนเป็น <strong>YOLOv8m</strong> หรือเพิ่ม Confidence threshold
          </div>
        </div>
      </div>
    </div>
    <footer class="bottombar">
      <span class="bb-dot"></span>
      <span>Detection Tool · EVD System</span>
      <span class="bb-sep">·</span>
      <span>NUOL · Computer Science · FYP 2025–26</span>
    </footer>
  </div>
  <script src="/static/detect.js?v=1"></script>
</body>
</html>
''', encoding='utf-8')
print('✅ /content/evd/templates/detect.html')

## 6️⃣  เขียน Static files (CSS / JS)

In [ ]:
from pathlib import Path
Path('/content/evd/static/landing.css').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/static/landing.css').write_text('''@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Lao:wght@400;600;700&family=Inter:wght@400;600;700&display=swap');

* { box-sizing: border-box; margin: 0; padding: 0; }

body {
  background: #f5f7fa;
  color: #0f172a;
  font-family: 'Noto Sans Lao', 'Inter', 'Segoe UI', system-ui, sans-serif;
  min-height: 100vh;
  display: flex;
  flex-direction: column;
}

/* ===== Topbar ===== */
.topbar {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 12px 40px;
  background: #ffffff;
  border-bottom: 1px solid #e2e6ed;
}
.topbar-logo {
  display: flex;
  align-items: center;
  gap: 14px;
}
.topbar-logo img {
  height: 56px;
  width: auto;
  display: block;
}
.topbar-logo-text {
  display: flex;
  flex-direction: column;
  gap: 2px;
}
.logo-name {
  font-size: 14px;
  font-weight: 700;
  color: #0f172a;
}
.logo-dept {
  font-size: 11px;
  color: #6b7280;
}
.topbar-badge {
  font-size: 11px;
  font-weight: 600;
  letter-spacing: 0.04em;
  padding: 4px 12px;
  border-radius: 20px;
  background: #eff6ff;
  color: #1A6BFF;
  border: 1px solid #bfdbfe;
}

/* ===== Main ===== */
main {
  flex: 1;
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  padding: 72px 24px 56px;
  gap: 56px;
}

/* ===== Hero ===== */
.hero {
  text-align: center;
  max-width: 600px;
}
.hero-eyebrow {
  display: inline-block;
  font-size: 11px;
  font-weight: 600;
  letter-spacing: 0.1em;
  text-transform: uppercase;
  color: #1A6BFF;
  background: #eff6ff;
  border: 1px solid #bfdbfe;
  border-radius: 20px;
  padding: 4px 14px;
  margin-bottom: 22px;
}
.hero-title {
  font-size: clamp(26px, 5vw, 44px);
  font-weight: 700;
  line-height: 1.2;
  letter-spacing: -0.02em;
  color: #0f172a;
  margin-bottom: 16px;
}
.hero-title span { color: #1A6BFF; }
.hero-desc {
  font-size: 15px;
  color: #6b7280;
  line-height: 1.8;
  margin-bottom: 36px;
}
.cta-btn {
  display: inline-flex;
  align-items: center;
  gap: 8px;
  background: #1A6BFF;
  color: #ffffff;
  font-size: 14px;
  font-weight: 600;
  letter-spacing: 0.02em;
  padding: 13px 28px;
  border-radius: 10px;
  text-decoration: none;
  transition: background 0.15s, transform 0.1s, box-shadow 0.15s;
  box-shadow: 0 2px 8px rgba(26,107,255,0.25);
}
.cta-btn:hover {
  background: #1558d6;
  transform: translateY(-1px);
  box-shadow: 0 4px 14px rgba(26,107,255,0.35);
}
.cta-btn:active { transform: translateY(0); }

/* ===== Feature cards ===== */
.features {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 16px;
  width: 100%;
  max-width: 820px;
}
.feat-card {
  background: #ffffff;
  border: 1px solid #e2e6ed;
  border-radius: 12px;
  padding: 24px 20px;
  display: flex;
  flex-direction: column;
  gap: 8px;
}
.feat-icon { font-size: 22px; margin-bottom: 4px; }
.feat-title { font-size: 13px; font-weight: 700; color: #0f172a; }
.feat-desc  { font-size: 12px; color: #6b7280; line-height: 1.7; }
.feat-tags  { display: flex; flex-wrap: wrap; gap: 5px; margin-top: 6px; }
.tag {
  font-size: 10px; font-weight: 600; letter-spacing: 0.04em;
  padding: 3px 8px; border-radius: 4px; text-transform: uppercase;
  background: #f1f5f9; border: 1px solid #e2e6ed; color: #6b7280;
}
.tag.amb    { color: #ea580c; background: #fff7ed; border-color: #fed7aa; }
.tag.fire   { color: #dc2626; background: #fef2f2; border-color: #fecaca; }
.tag.police { color: #2563eb; background: #eff6ff; border-color: #bfdbfe; }
.tag.green  { color: #16a34a; background: #f0fdf4; border-color: #bbf7d0; }

/* ===== Footer ===== */
footer {
  padding: 16px 40px;
  border-top: 1px solid #e2e6ed;
  background: #ffffff;
  text-align: center;
  font-size: 12px;
  color: #9ca3af;
  letter-spacing: 0.04em;
}

/* ===== Responsive ===== */
@media (max-width: 680px) {
  .features { grid-template-columns: 1fr; }
  main { padding: 48px 16px 40px; gap: 40px; }
  .topbar { padding: 12px 20px; }
  .topbar-logo img { height: 44px; }
}
''', encoding='utf-8')
print('✅ /content/evd/static/landing.css')

In [ ]:
from pathlib import Path
Path('/content/evd/static/style.css').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/static/style.css').write_text('''@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Lao:wght@400;600;700&family=Inter:wght@400;600;700&display=swap');

:root {
  --bg:     #f5f7fa;
  --panel:  #ffffff;
  --blue:   #1A6BFF;
  --green:  #16a34a;
  --red:    #dc2626;
  --amber:  #d97706;
  --text:   #0f172a;
  --muted:  #6b7280;
  --border: #e2e6ed;
  --card:   #f8fafc;
}

* { box-sizing: border-box; margin: 0; padding: 0; }

body {
  background: var(--bg);
  color: var(--text);
  font-family: 'Noto Sans Lao', 'Inter', 'Segoe UI', system-ui, sans-serif;
  min-height: 100vh;
  display: flex;
  align-items: center;
  justify-content: center;
  padding: 16px;
}

.dash {
  width: 100%;
  max-width: 1280px;
  background: var(--panel);
  border: 1px solid var(--border);
  border-radius: 12px;
  overflow: hidden;
}

/* ===== Topbar ===== */
.topbar {
  display: flex; align-items: center; justify-content: space-between;
  padding: 10px 20px;
  border-bottom: 1px solid var(--border);
  background: var(--panel);
}
.topbar-left { display: flex; align-items: center; gap: 12px; }
.topbar-logo-img { height: 48px; width: auto; display: block; }
.logo-divider { width: 1px; height: 32px; background: var(--border); }
.logo-dot {
  width: 8px; height: 8px; border-radius: 50%;
  background: var(--green); box-shadow: 0 0 8px var(--green);
  animation: pulse 2s infinite;
}
.logo-text { font-size: 13px; font-weight: 700; letter-spacing: 0.06em; color: var(--text); }
.logo-sub  { font-size: 11px; color: var(--muted); margin-top: 1px; }
.topbar-right { display: flex; align-items: center; gap: 14px; }
.status-badge {
  font-size: 11px; font-weight: 600; letter-spacing: 0.04em;
  padding: 4px 10px; border-radius: 20px;
  background: #f0fdf4; color: var(--green);
  border: 1px solid #bbf7d0;
}
.status-badge.warn {
  background: #fffbeb; color: var(--amber);
  border-color: #fde68a;
}
.clock { font-size: 12px; color: var(--muted); font-family: ui-monospace, monospace; }

/* ===== Layout ===== */
.main { display: grid; grid-template-columns: 1fr 230px; min-height: 540px; }

.cam-grid {
  display: grid; grid-template-columns: 1fr 1fr; gap: 2px;
  background: var(--border); padding: 2px;
}
.cam-cell { position: relative; background: #0f1117; overflow: hidden; aspect-ratio: 16/9; }
.cam-feed { width: 100%; height: 100%; object-fit: cover; display: block; }
.cam-label {
  position: absolute; top: 8px; left: 8px;
  font-size: 10px; font-weight: 600; letter-spacing: 0.06em;
  color: rgba(255,255,255,0.9);
  background: rgba(0,0,0,0.55); padding: 3px 7px; border-radius: 4px;
}
.cam-corner { position: absolute; top: 8px; right: 8px; display: flex; align-items: center; gap: 5px; }
.rec-dot { width: 6px; height: 6px; border-radius: 50%; background: var(--red); animation: pulse 1s infinite; }
.rec-text { font-size: 9px; color: rgba(255,255,255,0.7); letter-spacing: 0.05em; }

/* ===== Sidebar ===== */
.sidebar { background: var(--panel); border-left: 1px solid var(--border); display: flex; flex-direction: column; }
.sidebar-section { padding: 14px; border-bottom: 1px solid var(--border); }
.sidebar-section.grow { flex: 1; overflow: hidden; }
.sidebar-title {
  font-size: 10px; font-weight: 600; letter-spacing: 0.08em;
  color: var(--muted); margin-bottom: 10px; text-transform: uppercase;
}

.traffic-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
.traffic-light {
  display: flex; flex-direction: column; align-items: center; gap: 5px;
  background: var(--card); border: 1px solid var(--border);
  border-radius: 8px; padding: 10px 6px;
}
.tl-label { font-size: 9px; color: var(--muted); letter-spacing: 0.04em; }
.tl-box {
  width: 32px; background: #f1f5f9; border-radius: 6px; padding: 4px;
  display: flex; flex-direction: column; gap: 3px; border: 1px solid var(--border);
}
.tl-bulb { width: 24px; height: 24px; border-radius: 50%; margin: 0 auto; }
.tl-bulb.r { background: #fecaca; }
.tl-bulb.y { background: #fef3c7; }
.tl-bulb.g { background: #bbf7d0; }
.tl-bulb.r.on { background: var(--red);   box-shadow: 0 0 10px var(--red); }
.tl-bulb.y.on { background: #f59e0b;      box-shadow: 0 0 10px #f59e0b; }
.tl-bulb.g.on { background: var(--green); box-shadow: 0 0 10px var(--green); }
.tl-status { font-size: 9px; font-weight: 600; letter-spacing: 0.04em; }
.tl-status.stop  { color: var(--red); }
.tl-status.clear { color: var(--green); animation: blink 0.8s infinite; }

.alert-list { display: flex; flex-direction: column; gap: 6px; max-height: 200px; overflow-y: auto; }
.empty-hint { font-size: 11px; color: var(--muted); padding: 8px 2px; }
.alert-item {
  display: flex; align-items: flex-start; gap: 8px;
  background: var(--card); border-radius: 6px; padding: 8px 10px;
  border-left: 3px solid var(--muted);
}
.alert-item.ambulance { border-left-color: #ea580c; }
.alert-item.firetruck { border-left-color: var(--red); }
.alert-item.police    { border-left-color: #2563eb; }
.alert-type { font-size: 11px; font-weight: 600; text-transform: capitalize; color: var(--text); }
.alert-meta { font-size: 10px; color: var(--muted); margin-top: 1px; }

.stat-row { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 6px; }
.stat-card { background: var(--card); border: 1px solid var(--border); border-radius: 6px; padding: 8px; text-align: center; }
.stat-num { font-size: 18px; font-weight: 700; font-family: ui-monospace, monospace; }
.stat-lbl { font-size: 9px; color: var(--muted); margin-top: 2px; letter-spacing: 0.04em; }
.stat-num.amb    { color: #ea580c; }
.stat-num.fire   { color: var(--red); }
.stat-num.police { color: #2563eb; }

/* ===== Bottombar ===== */
.bottombar {
  display: flex; align-items: center; gap: 12px;
  padding: 8px 16px; border-top: 1px solid var(--border);
  background: var(--card); font-size: 11px; color: var(--muted);
}
.bb-dot { width: 6px; height: 6px; border-radius: 50%; background: var(--green); }
.bb-sep { color: var(--border); }
.model-tag { color: var(--blue); font-family: ui-monospace, monospace; font-size: 10px; }

/* ===== Inline controls (model selector + conf slider in topbar) ===== */
.ctrl-inline {
  display: flex; align-items: center; gap: 6px;
}
.ctrl-inline-label {
  font-size: 10px; font-weight: 600; color: var(--muted);
  letter-spacing: 0.04em; white-space: nowrap;
}
.ctrl-inline-select {
  font-size: 11px; padding: 3px 6px; border-radius: 6px;
  border: 1px solid var(--border); background: var(--card); color: var(--text);
  cursor: pointer;
}
.ctrl-inline-slider {
  width: 80px; cursor: pointer; accent-color: var(--blue);
}
.conf-ctrl { gap: 4px; }
.nav-pill {
  font-size: 11px; font-weight: 600; padding: 4px 10px;
  border-radius: 20px; background: var(--blue); color: #fff;
  text-decoration: none; white-space: nowrap;
}
.nav-pill:hover { opacity: 0.85; }
.bb-params { color: var(--muted); font-size: 10px; }

@keyframes pulse { 0%,100% { opacity: 1; } 50% { opacity: 0.4; } }
@keyframes blink { 0%,100% { opacity: 1; } 50% { opacity: 0.4; } }

@media (max-width: 820px) {
  .main { grid-template-columns: 1fr; }
  .sidebar { border-left: none; border-top: 1px solid var(--border); }
  .topbar-right { flex-wrap: wrap; gap: 8px; }
  .ctrl-inline-slider { width: 60px; }
  .conf-ctrl { display: none; }   /* ซ่อน conf บน mobile — ยังปรับได้ใน detect page */
}
''', encoding='utf-8')
print('✅ /content/evd/static/style.css')

In [ ]:
from pathlib import Path
Path('/content/evd/static/app.js').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/static/app.js').write_text('''// app.js — Dashboard: อัปเดตไฟจราจร / log / ตัวนับ + จัดการ model selector / conf slider

function pad(n) { return String(n).padStart(2, "0"); }

function tickClock() {
  const d = new Date();
  document.getElementById("clk").textContent =
    `${pad(d.getHours())}:${pad(d.getMinutes())}:${pad(d.getSeconds())}`;
}
setInterval(tickClock, 1000);
tickClock();

const ICONS = { ambulance: "🚑", firetruck: "🚒", police: "🚔" };

// ===== Model selector & Confidence slider =====

const modelSelect = document.getElementById("modelSelect");
const confSlider  = document.getElementById("confSlider");
const confDisplay = document.getElementById("confDisplay");

async function loadSettings() {
  try {
    const res  = await fetch("/api/settings");
    const data = await res.json();
    modelSelect.value   = data.model;
    confSlider.value    = data.conf;
    confDisplay.textContent = parseFloat(data.conf).toFixed(2);
  } catch (_) {}
}

async function pushSettings() {
  await fetch("/api/settings", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({
      model: modelSelect.value,
      conf:  parseFloat(confSlider.value),
    }),
  });
}

modelSelect.addEventListener("change", pushSettings);

confSlider.addEventListener("input", () => {
  confDisplay.textContent = parseFloat(confSlider.value).toFixed(2);
});
confSlider.addEventListener("change", pushSettings);

loadSettings();

// ===== Stats polling =====

async function refreshStats() {
  try {
    const res  = await fetch("/stats");
    const data = await res.json();

    updateSignals(data.signals);
    updateCounts(data.counts);
    updateLog(data.log);

    document.getElementById("inf").textContent =
      data.infer_ms ? `${data.infer_ms} ms` : "--";

    if (data.model_info) {
      const mi = data.model_info;
      const el = document.getElementById("bottomModel");
      if (el) el.textContent = mi.name;
      const ep = document.getElementById("bottomParams");
      if (ep) ep.textContent = mi.params;
      const ml = document.getElementById("modelLabel");
      if (ml) ml.textContent = mi.name;
    }
  } catch (_) {}
}

function updateSignals(signals) {
  document.querySelectorAll(".traffic-light").forEach((el) => {
    const cam   = el.dataset.cam;
    const sig   = signals[cam] || "STOP";
    const clear = sig === "CLEAR";
    el.querySelector(".tl-bulb.r").classList.toggle("on", !clear);
    el.querySelector(".tl-bulb.g").classList.toggle("on",  clear);
    const status = el.querySelector(".tl-status");
    status.textContent = clear ? "CLEAR" : "STOP";
    status.className   = "tl-status " + (clear ? "clear" : "stop");
  });
}

function updateCounts(counts) {
  document.getElementById("cntAmb").textContent    = counts.ambulance || 0;
  document.getElementById("cntFire").textContent   = counts.firetruck || 0;
  document.getElementById("cntPolice").textContent = counts.police    || 0;
}

function updateLog(log) {
  const list = document.getElementById("alertList");
  if (!log || log.length === 0) {
    list.innerHTML = '<div class="empty-hint">ລໍຖ້າການກວດຈັບ...</div>';
    return;
  }
  list.innerHTML = log.map((item) => `
    <div class="alert-item ${item.name}">
      <div class="alert-icon">${ICONS[item.name] || "🚨"}</div>
      <div class="alert-body">
        <div class="alert-type">${item.name}</div>
        <div class="alert-meta">${item.cam} · ${item.conf}% · ${item.t}</div>
      </div>
    </div>
  `).join("");
}

setInterval(refreshStats, 1000);
refreshStats();
''', encoding='utf-8')
print('✅ /content/evd/static/app.js')

In [ ]:
from pathlib import Path
Path('/content/evd/static/detect.css').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/static/detect.css').write_text('''/* detect.css — Detection Tool page styles (shares :root from style.css) */

.detect-dash { display: flex; flex-direction: column; min-height: 100vh; }

/* ===== Shared controls bar ===== */
.detect-controls {
  display: flex; align-items: center; gap: 24px;
  padding: 10px 20px; border-bottom: 1px solid var(--border);
  background: var(--card); flex-wrap: wrap;
}
.detect-ctrl-group { display: flex; align-items: center; gap: 10px; }
.detect-ctrl-label {
  font-size: 12px; font-weight: 600; color: var(--muted);
  letter-spacing: 0.04em; white-space: nowrap;
}
.detect-ctrl-select {
  font-size: 12px; padding: 5px 10px; border-radius: 8px;
  border: 1px solid var(--border); background: var(--panel); color: var(--text);
  cursor: pointer; min-width: 240px;
}
.detect-slider { width: 160px; cursor: pointer; accent-color: var(--blue); }
.conf-badge {
  display: inline-block; min-width: 34px; text-align: center;
  font-family: ui-monospace, monospace; font-size: 12px;
  font-weight: 700; color: var(--blue);
}

/* ===== Tabs ===== */
.tab-bar {
  display: flex; gap: 0; border-bottom: 2px solid var(--border);
  background: var(--panel); padding: 0 16px;
}
.tab-btn {
  padding: 10px 18px; font-size: 13px; font-weight: 600;
  border: none; background: none; cursor: pointer; color: var(--muted);
  border-bottom: 2px solid transparent; margin-bottom: -2px;
  transition: color 0.15s, border-color 0.15s;
}
.tab-btn:hover { color: var(--blue); }
.tab-btn.active { color: var(--blue); border-bottom-color: var(--blue); }

/* ===== Tab body & panes ===== */
.tab-body { flex: 1; padding: 20px; background: var(--bg); }
.tab-pane { max-width: 860px; margin: 0 auto; }
.tab-pane.hidden { display: none; }

/* ===== Upload zone ===== */
.upload-zone {
  border: 2px dashed var(--border); border-radius: 12px;
  padding: 40px 20px; text-align: center; cursor: pointer;
  background: var(--panel); transition: border-color 0.2s, background 0.2s;
}
.upload-zone:hover, .upload-zone.drag-over {
  border-color: var(--blue); background: #eff6ff;
}
.upload-zone.has-file { border-color: var(--green); background: #f0fdf4; }
.upload-icon { font-size: 40px; margin-bottom: 8px; }
.upload-text { font-size: 14px; font-weight: 600; color: var(--text); margin-bottom: 4px; }
.upload-hint { font-size: 12px; color: var(--muted); }

/* ===== Actions ===== */
.tab-actions { margin: 14px 0; display: flex; gap: 10px; align-items: center; }
.action-btn {
  padding: 9px 22px; border-radius: 8px; font-size: 13px; font-weight: 600;
  border: none; cursor: pointer; background: var(--blue); color: #fff;
  transition: opacity 0.15s;
}
.action-btn:hover { opacity: 0.88; }
.action-btn:disabled { opacity: 0.4; cursor: not-allowed; }
.action-btn.danger { background: var(--red); }

/* ===== Progress ===== */
.progress-row {
  display: flex; align-items: center; gap: 10px;
  font-size: 13px; color: var(--muted); padding: 8px 0;
}
.progress-row.hidden { display: none; }
.spinner {
  width: 16px; height: 16px; border-radius: 50%;
  border: 2px solid var(--border); border-top-color: var(--blue);
  animation: spin 0.7s linear infinite; display: inline-block;
}
@keyframes spin { to { transform: rotate(360deg); } }

/* ===== Result area ===== */
.result-area { margin-top: 16px; }
.result-area.hidden { display: none; }
.result-meta {
  font-size: 12px; color: var(--muted); padding: 8px 12px;
  background: var(--panel); border: 1px solid var(--border);
  border-radius: 8px; margin-bottom: 12px; line-height: 1.7;
}
.result-meta strong { color: var(--blue); }
.result-layout { display: flex; gap: 16px; flex-wrap: wrap; }
.result-img {
  max-width: 100%; border-radius: 8px;
  border: 1px solid var(--border); flex: 1 1 400px;
}

/* ===== Detection list ===== */
.det-list { flex: 0 0 200px; display: flex; flex-direction: column; gap: 6px; }
.det-item {
  padding: 8px 10px; border-radius: 6px; font-size: 12px;
  background: var(--card); border-left: 3px solid var(--muted);
}
.det-item.ambulance { border-left-color: #ea580c; }
.det-item.firetruck { border-left-color: var(--red); }
.det-item.police    { border-left-color: #2563eb; }
.det-name { font-weight: 700; text-transform: capitalize; color: var(--text); }
.det-conf { color: var(--muted); font-size: 11px; }

/* ===== Video stats cards ===== */
.video-stats { display: flex; gap: 12px; flex-wrap: wrap; margin-bottom: 16px; }
.vstat-card {
  flex: 1 1 120px; background: var(--panel); border: 1px solid var(--border);
  border-radius: 8px; padding: 12px 16px; text-align: center;
}
.vstat-num { font-size: 22px; font-weight: 700; color: var(--blue); font-family: ui-monospace, monospace; }
.vstat-lbl { font-size: 10px; color: var(--muted); margin-top: 2px; letter-spacing: 0.06em; }
.sample-label { font-size: 11px; font-weight: 600; color: var(--muted); margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.06em; }

/* ===== Webcam ===== */
.webcam-wrap { display: flex; flex-direction: column; gap: 12px; }
.webcam-view {
  position: relative; width: 100%; max-width: 720px;
  background: #111; border-radius: 10px; overflow: hidden;
  aspect-ratio: 4/3;
}
.webcam-view video,
.webcam-view canvas {
  position: absolute; inset: 0; width: 100%; height: 100%;
}
.webcam-view canvas { pointer-events: none; }
.wcam-overlay {
  position: absolute; z-index: 10; font-family: ui-monospace, monospace;
  font-size: 13px; font-weight: 700; padding: 3px 8px; border-radius: 4px;
  background: rgba(0,0,0,0.55); color: #fff;
}
.fps-overlay   { top: 8px; left: 8px; color: #4ade80; }
.model-overlay { top: 8px; right: 8px; color: #93c5fd; font-size: 11px; }
.webcam-actions { display: flex; gap: 10px; }

/* ===== Messages ===== */
.error-box {
  padding: 12px 16px; border-radius: 8px; font-size: 13px;
  background: #fef2f2; border: 1px solid #fecaca; color: var(--red);
}
.hint-box {
  padding: 12px 16px; border-radius: 8px; font-size: 13px;
  background: #fffbeb; border: 1px solid #fde68a; color: #92400e;
}
.error-box.hidden, .hint-box.hidden { display: none; }

@media (max-width: 640px) {
  .detect-controls { flex-direction: column; align-items: flex-start; }
  .result-layout { flex-direction: column; }
  .det-list { flex: none; }
}
''', encoding='utf-8')
print('✅ /content/evd/static/detect.css')

In [ ]:
from pathlib import Path
Path('/content/evd/static/detect.js').parent.mkdir(parents=True, exist_ok=True)
Path('/content/evd/static/detect.js').write_text('''// detect.js — Detection Tool page logic

// ===== Clock =====
(function tickClock() {
  const p = n => String(n).padStart(2, "0");
  const d = new Date();
  const el = document.getElementById("clk");
  if (el) el.textContent = `${p(d.getHours())}:${p(d.getMinutes())}:${p(d.getSeconds())}`;
  setTimeout(tickClock, 1000);
})();

// ===== Shared controls =====
const modelSelect = document.getElementById("modelSelect");
const confSlider  = document.getElementById("confSlider");
const confVal     = document.getElementById("confVal");

confSlider.addEventListener("input", () => {
  confVal.textContent = parseFloat(confSlider.value).toFixed(2);
});

function getModel() { return modelSelect.value; }
function getConf()  { return parseFloat(confSlider.value); }

// ===== Tab switching =====
document.querySelectorAll(".tab-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    const target = btn.dataset.tab;
    document.querySelectorAll(".tab-btn").forEach(b => b.classList.remove("active"));
    document.querySelectorAll(".tab-pane").forEach(p => p.classList.add("hidden"));
    btn.classList.add("active");
    document.getElementById(`tab-${target}`).classList.remove("hidden");

    // หยุดกล้องถ้าออกจาก webcam tab
    if (target !== "webcam" && webcamRunning) stopWebcam();
  });
});

// ===== Helpers =====
const DET_COLORS = { ambulance: "#ea580c", firetruck: "#dc2626", police: "#2563eb" };
function detColor(name) { return DET_COLORS[name.toLowerCase()] || "#16a34a"; }

function makeDropZone(zoneEl, inputEl, onFileReady) {
  zoneEl.addEventListener("click", () => inputEl.click());
  inputEl.addEventListener("change", () => {
    if (inputEl.files[0]) {
      zoneEl.classList.add("has-file");
      zoneEl.querySelector(".upload-text").textContent = inputEl.files[0].name;
      onFileReady(inputEl.files[0]);
    }
  });
  zoneEl.addEventListener("dragover", e => { e.preventDefault(); zoneEl.classList.add("drag-over"); });
  zoneEl.addEventListener("dragleave", () => zoneEl.classList.remove("drag-over"));
  zoneEl.addEventListener("drop", e => {
    e.preventDefault();
    zoneEl.classList.remove("drag-over");
    const f = e.dataTransfer.files[0];
    if (f) {
      zoneEl.classList.add("has-file");
      zoneEl.querySelector(".upload-text").textContent = f.name;
      onFileReady(f);
    }
  });
}

// ===========================================================
// IMAGE TAB
// ===========================================================
const imageFile      = document.getElementById("imageFile");
const imageDropZone  = document.getElementById("imageDropZone");
const imageDetectBtn = document.getElementById("imageDetectBtn");
const imageResult    = document.getElementById("imageResult");
const imageMeta      = document.getElementById("imageMeta");
const resultImg      = document.getElementById("resultImg");
const imageDetList   = document.getElementById("imageDetList");

let imageFileObj = null;

makeDropZone(imageDropZone, imageFile, f => {
  imageFileObj = f;
  imageDetectBtn.disabled = false;
  imageResult.classList.add("hidden");
});

imageDetectBtn.addEventListener("click", async () => {
  if (!imageFileObj) return;
  imageDetectBtn.disabled = true;
  imageDetectBtn.textContent = "กำลังตรวจจับ...";
  imageResult.classList.add("hidden");

  const fd = new FormData();
  fd.append("file", imageFileObj);
  fd.append("model_name", getModel());
  fd.append("conf", getConf());

  const t0 = performance.now();
  try {
    const res  = await fetch("/predict/image", { method: "POST", body: fd });
    const data = await res.json();
    const elapsed = Math.round(performance.now() - t0);

    if (data.error) { alert(data.error); return; }

    resultImg.src = `data:image/jpeg;base64,${data.image}`;
    imageMeta.innerHTML =
      `โมเดล: <strong>${data.model_name}</strong> &nbsp;·&nbsp; ${data.model_params} &nbsp;·&nbsp; ` +
      `ประมวลผลใน <strong>${data.infer_ms} ms</strong> &nbsp;·&nbsp; ` +
      `พบ <strong>${data.detections.length}</strong> วัตถุ`;

    imageDetList.innerHTML = data.detections.length === 0
      ? '<div class="det-item">ไม่พบวัตถุ</div>'
      : data.detections.map(d => `
          <div class="det-item ${d.name.toLowerCase()}">
            <div class="det-name">${d.name}</div>
            <div class="det-conf">${Math.round(d.conf * 100)}% confidence</div>
          </div>`).join("");

    imageResult.classList.remove("hidden");
  } catch (e) {
    alert("เกิดข้อผิดพลาด: " + e.message);
  } finally {
    imageDetectBtn.disabled = false;
    imageDetectBtn.textContent = "ตรวจจับ";
  }
});

// ===========================================================
// VIDEO TAB
// ===========================================================
const videoFile      = document.getElementById("videoFile");
const videoDropZone  = document.getElementById("videoDropZone");
const videoDetectBtn = document.getElementById("videoDetectBtn");
const videoProgress  = document.getElementById("videoProgress");
const progressText   = document.getElementById("progressText");
const videoResult    = document.getElementById("videoResult");
const videoMeta      = document.getElementById("videoMeta");
const videoStats     = document.getElementById("videoStats");
const videoSampleWrap = document.getElementById("videoSampleWrap");
const videoSampleImg  = document.getElementById("videoSampleImg");

let videoFileObj = null;

makeDropZone(videoDropZone, videoFile, f => {
  videoFileObj = f;
  videoDetectBtn.disabled = false;
  videoResult.classList.add("hidden");
});

videoDetectBtn.addEventListener("click", async () => {
  if (!videoFileObj) return;
  videoDetectBtn.disabled = true;
  videoDetectBtn.textContent = "กำลังประมวลผล...";
  videoResult.classList.add("hidden");
  videoProgress.classList.remove("hidden");
  progressText.textContent = "กำลังส่งและประมวลผลวิดีโอ... อาจใช้เวลาสักครู่";

  const fd = new FormData();
  fd.append("file", videoFileObj);
  fd.append("model_name", getModel());
  fd.append("conf", getConf());

  try {
    const res  = await fetch("/predict/video", { method: "POST", body: fd });
    const data = await res.json();

    if (data.error) { alert(data.error); return; }

    videoMeta.innerHTML =
      `โมเดล: <strong>${data.model_name}</strong> &nbsp;·&nbsp; ${data.model_params}`;

    const totalSec = (data.total_ms / 1000).toFixed(1);
    const detTotal = Object.values(data.detections_summary).reduce((a, b) => a + b, 0);

    videoStats.innerHTML = `
      <div class="vstat-card">
        <div class="vstat-num">${data.frame_count.toLocaleString()}</div>
        <div class="vstat-lbl">FRAMES PROCESSED</div>
      </div>
      <div class="vstat-card">
        <div class="vstat-num">${totalSec}s</div>
        <div class="vstat-lbl">TOTAL TIME</div>
      </div>
      <div class="vstat-card">
        <div class="vstat-num">${data.avg_ms_per_frame}</div>
        <div class="vstat-lbl">ms / FRAME</div>
      </div>
      <div class="vstat-card">
        <div class="vstat-num">${detTotal}</div>
        <div class="vstat-lbl">DETECTIONS</div>
      </div>
    ` + Object.entries(data.detections_summary).map(([name, count]) => `
      <div class="vstat-card">
        <div class="vstat-num" style="color:${detColor(name)}">${count}</div>
        <div class="vstat-lbl">${name.toUpperCase()}</div>
      </div>`).join("");

    if (data.sample_frame) {
      videoSampleImg.src = `data:image/jpeg;base64,${data.sample_frame}`;
      videoSampleWrap.classList.remove("hidden");
    } else {
      videoSampleWrap.classList.add("hidden");
    }

    videoResult.classList.remove("hidden");
  } catch (e) {
    alert("เกิดข้อผิดพลาด: " + e.message);
  } finally {
    videoDetectBtn.disabled = false;
    videoDetectBtn.textContent = "ประมวลผลวิดีโอ";
    videoProgress.classList.add("hidden");
  }
});

// ===========================================================
// WEBCAM TAB
// ===========================================================
const webcamVideo   = document.getElementById("webcamVideo");
const webcamCanvas  = document.getElementById("webcamCanvas");
const startBtn      = document.getElementById("startWebcam");
const stopBtn       = document.getElementById("stopWebcam");
const fpsOverlay    = document.getElementById("fpsOverlay");
const modelOverlay  = document.getElementById("modelOverlay");
const webcamError   = document.getElementById("webcamError");
const webcamHint    = document.getElementById("webcamHint");

let webcamRunning   = false;
let webcamStream    = null;
let fpsSamples      = [];
const SEND_WIDTH    = 640;
const MIN_INTERVAL  = 300;  // ms — ส่ง frame อย่างช้าที่สุด ~3.3 fps

async function startWebcam() {
  webcamError.classList.add("hidden");
  webcamHint.classList.add("hidden");
  try {
    webcamStream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 1280 }, height: { ideal: 720 }, facingMode: "environment" },
      audio: false,
    });
    webcamVideo.srcObject = webcamStream;
    await webcamVideo.play();
    webcamRunning = true;
    startBtn.disabled = true;
    stopBtn.disabled  = false;
    fpsOverlay.textContent    = "-- FPS";
    modelOverlay.textContent  = "--";
    fpsSamples = [];
    webcamDetectionLoop();
  } catch (err) {
    let msg = "ไม่สามารถเปิดกล้องได้";
    if (err.name === "NotAllowedError" || err.name === "PermissionDeniedError") {
      msg = "❌ ไม่ได้รับอนุญาตให้ใช้กล้อง — กรุณาอนุญาตการเข้าถึงกล้องในเบราว์เซอร์แล้วลองใหม่";
    } else if (err.name === "NotFoundError") {
      msg = "❌ ไม่พบกล้อง — กรุณาเชื่อมต่อเว็บแคมแล้วลองใหม่";
    } else if (err.name === "NotReadableError") {
      msg = "❌ กล้องกำลังถูกใช้งานโดยแอปอื่น — กรุณาปิดแอปอื่นแล้วลองใหม่";
    } else {
      msg = `❌ ${err.name}: ${err.message}`;
    }
    webcamError.textContent = msg;
    webcamError.classList.remove("hidden");
  }
}

function stopWebcam() {
  webcamRunning = false;
  if (webcamStream) {
    webcamStream.getTracks().forEach(t => t.stop());
    webcamStream = null;
  }
  webcamVideo.srcObject = null;
  const ctx = webcamCanvas.getContext("2d");
  ctx.clearRect(0, 0, webcamCanvas.width, webcamCanvas.height);
  fpsOverlay.textContent = "-- FPS";
  startBtn.disabled = false;
  stopBtn.disabled  = true;
}

async function webcamDetectionLoop() {
  if (!webcamRunning) return;

  const t0 = performance.now();

  try {
    // 1) Capture current video frame
    const ratio  = webcamVideo.videoHeight / webcamVideo.videoWidth;
    const sendH  = Math.round(SEND_WIDTH * ratio) || 480;
    const offscreen = document.createElement("canvas");
    offscreen.width  = SEND_WIDTH;
    offscreen.height = sendH;
    offscreen.getContext("2d").drawImage(webcamVideo, 0, 0, SEND_WIDTH, sendH);
    const b64 = offscreen.toDataURL("image/jpeg", 0.75);

    // 2) Send to backend
    const resp = await fetch("/predict/webcam_frame", {
      method:  "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({
        image:      b64,
        model_name: getModel(),
        conf:       getConf(),
      }),
    });
    const data = await resp.json();

    // 3) Update FPS (rolling average of last 8 samples)
    const elapsed = performance.now() - t0;
    fpsSamples.push(elapsed);
    if (fpsSamples.length > 8) fpsSamples.shift();
    const avgMs = fpsSamples.reduce((a, b) => a + b, 0) / fpsSamples.length;
    const fps   = Math.round(1000 / avgMs);
    fpsOverlay.textContent   = `${fps} FPS · ${Math.round(data.infer_ms)}ms`;
    modelOverlay.textContent = `${data.model_name} · ${data.model_params}`;

    // แสดง hint เมื่อ FPS ต่ำ
    if (fps < 3) webcamHint.classList.remove("hidden");

    // 4) Draw detections on canvas overlay
    drawDetections(data.detections, SEND_WIDTH, sendH);

  } catch (e) {
    // ข้ามถ้าเน็ตหรือเซิร์ฟเวอร์มีปัญหา
    console.warn("webcam frame error:", e);
  }

  // 5) Wait remainder of MIN_INTERVAL before next frame
  const spent = performance.now() - t0;
  const wait  = Math.max(0, MIN_INTERVAL - spent);
  if (webcamRunning) setTimeout(webcamDetectionLoop, wait);
}

function drawDetections(dets, srcW, srcH) {
  const canvas = webcamCanvas;
  const ctx    = canvas.getContext("2d");
  canvas.width  = webcamVideo.clientWidth  || 640;
  canvas.height = webcamVideo.clientHeight || 480;
  ctx.clearRect(0, 0, canvas.width, canvas.height);

  if (!dets || dets.length === 0) return;

  const scaleX = canvas.width  / srcW;
  const scaleY = canvas.height / srcH;

  ctx.font = "bold 13px Inter, 'Segoe UI', sans-serif";

  for (const det of dets) {
    const [x1, y1, x2, y2] = det.box;
    const color = detColor(det.name);
    const label = `${det.name.toUpperCase()} ${Math.round(det.conf * 100)}%`;

    // Bounding box
    ctx.strokeStyle = color;
    ctx.lineWidth   = 2;
    ctx.strokeRect(x1 * scaleX, y1 * scaleY, (x2 - x1) * scaleX, (y2 - y1) * scaleY);

    // Label background
    const tw = ctx.measureText(label).width;
    ctx.fillStyle = color;
    ctx.fillRect(x1 * scaleX, y1 * scaleY - 20, tw + 10, 20);

    // Label text
    ctx.fillStyle = "#ffffff";
    ctx.fillText(label, x1 * scaleX + 5, y1 * scaleY - 5);
  }
}

startBtn.addEventListener("click", startWebcam);
stopBtn.addEventListener("click",  stopWebcam);
''', encoding='utf-8')
print('✅ /content/evd/static/detect.js')

## 7️⃣  อัปโหลด Model Files

อัปโหลด `best_x.pt` (YOLOv8x) และ `best_m.pt` (YOLOv8m)
ถ้ายังไม่มีไฟล์ model ระบบจะ **ดาวน์โหลด pretrained** โดยอัตโนมัติ (ใช้ได้แต่ไม่ใช่ custom model ของคุณ)


In [ ]:
from google.colab import files
import shutil

print("เลือกไฟล์ best_x.pt (YOLOv8x ที่เทรนเอง) — กด Cancel ถ้ายังไม่มี")
try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f'/content/evd/models/best_x.pt'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f"✅ บันทึก {fname} → {dest}  ({len(data)/1e6:.1f} MB)")
except Exception as e:
    print(f"⚠️  ข้าม best_x.pt ({e}) — จะใช้ pretrained yolov8x.pt แทน")


In [ ]:
from google.colab import files

print("เลือกไฟล์ best_m.pt (YOLOv8m ที่เทรนเอง) — กด Cancel ถ้ายังไม่มี")
try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f'/content/evd/models/best_m.pt'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f"✅ บันทึก {fname} → {dest}  ({len(data)/1e6:.1f} MB)")
except Exception as e:
    print(f"⚠️  ข้าม best_m.pt ({e}) — จะใช้ pretrained yolov8m.pt แทน")


## 8️⃣  อัปโหลดวิดีโอ (optional)

ถ้าไม่อัปโหลด กล้องจะแสดง **NO SIGNAL** (ยังใช้ Detection Tool ได้ปกติ)
ชื่อไฟล์ที่รองรับ: `north.mp4`, `east.mp4`, `south.mp4`, `west.mp4`


In [ ]:
from google.colab import files
import os

VIDEO_NAMES = ['north.mp4', 'east.mp4', 'south.mp4', 'west.mp4']
print(f"เลือกไฟล์วิดีโอ (กด Cancel เพื่อข้าม)  ชื่อที่ต้องการ: {VIDEO_NAMES}")
try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        # ถ้าชื่อตรง → บันทึกตรงตำแหน่ง, ถ้าไม่ตรง → ถามให้ map
        if fname in VIDEO_NAMES:
            dest = f'/content/evd/videos/{fname}'
        else:
            dest = f'/content/evd/videos/{fname}'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f"✅ {fname} → {dest}  ({len(data)/1e6:.1f} MB)")
except Exception as e:
    print(f"⚠️  ข้ามวิดีโอ ({e})")

# แสดงไฟล์ที่มี
existing = [f for f in VIDEO_NAMES if os.path.exists(f'/content/evd/videos/{f}')]
missing  = [f for f in VIDEO_NAMES if not os.path.exists(f'/content/evd/videos/{f}')]
if existing: print(f"\n📹 วิดีโอที่พร้อมใช้: {existing}")
if missing:  print(f"⚠️  ไม่มีไฟล์ (จะขึ้น NO SIGNAL): {missing}")


## 9️⃣  เริ่มเซิร์ฟเวอร์ + Public URL

เซลล์นี้จะ:
1. โหลดโมเดลทั้ง 2 ตัว (อาจใช้เวลา 1–2 นาที ถ้าต้องดาวน์โหลด pretrained)
2. เปิด FastAPI server บนพอร์ต 8000
3. สร้าง public URL ผ่าน **ngrok** หรือ **cloudflared**


In [ ]:
# ────────────────────────────────────────────────────────────
# ตัวเลือก A: ngrok (แนะนำ — ต้องมี authtoken จาก ngrok.com)
# ────────────────────────────────────────────────────────────
NGROK_TOKEN = ''  # ← ใส่ token ของคุณที่นี่ (หรือเว้นว่างเพื่อใช้ cloudflared แทน)

import subprocess, threading, time, os, sys

os.chdir('/content/evd')
sys.path.insert(0, '/content/evd')

def start_server():
    subprocess.Popen(
        ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/evd'
    )

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(8)   # รอ models โหลด
print("🚀 Server started on port 8000")


In [ ]:
import os, re, subprocess, threading, time

NGROK_TOKEN = ''  # ← ถ้ามี token ใส่ตรงนี้

if NGROK_TOKEN:
    # ── ngrok ──
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000, bind_tls=True)
    url    = tunnel.public_url
    print(f"\n{'='*60}")
    print(f"  🌐  Public URL (ngrok):  {url}")
    print(f"  📊  Dashboard:           {url}/dashboard")
    print(f"  🔬  Detection Tool:      {url}/detect")
    print(f"{'='*60}\n")
else:
    # ── cloudflared (ไม่ต้องมี account) ──
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
    result = {"url": None}
    def run_cf():
        proc = subprocess.Popen(
            ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        for line in proc.stdout:
            line = line.decode()
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                result["url"] = m.group(0)
                print(f"\n{'='*60}")
                print(f"  🌐  Public URL (cloudflare):  {result['url']}")
                print(f"  📊  Dashboard:   {result['url']}/dashboard")
                print(f"  🔬  Detection:   {result['url']}/detect")
                print(f"{'='*60}\n")
                break
    threading.Thread(target=run_cf, daemon=True).start()
    for _ in range(30):
        if result["url"]: break
        time.sleep(1)
    if not result["url"]:
        print("⚠️  Tunnel ยังไม่พร้อม — รอสักครู่แล้วลอง cell นี้ใหม่")


## 🔟  ตรวจสอบสถานะ (optional)

In [ ]:
import requests, time
time.sleep(2)
try:
    r = requests.get('http://localhost:8000/stats', timeout=5)
    print("✅ Server OK:", r.status_code)
    import json
    data = r.json()
    print(f"   Model: {data.get('current_model','?')} | Conf: {data.get('current_conf','?')}")
    info = data.get('model_info', {})
    print(f"   {info.get('name','?')} · {info.get('params','?')}")
except Exception as e:
    print(f"❌ Server error: {e} — รอเพิ่มอีกสักครู่แล้วลองใหม่")
